**Fig. 3, step 2: train the from-scratch lateral model, conservative regime, 11 stacks**

Needs a GPU.

In [ ]:
# Working directory: data/fig3/ under the repository root, where the Fig. 3 deposit folder is downloaded
# (decomposed_2D/, bootstrap_*_models/, domain-specialized_*/, benchmark_outputs*/, metrics_outputs/).
# The 11 ground-truth stacks are read from ../ground_truth/zebrafish/{development,held_out}/.
# Set the EPICYCLE_ANALYSES_DATA environment variable to keep the deposit somewhere else.
import os, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))   # the notebook's own folder, for benchmarking_scripts/
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment.yml").exists() and (p / "data").is_dir()), None)
assert _root is not None, "start Jupyter from inside the EpiCycle-analyses repository"
os.chdir(Path(os.environ.get("EPICYCLE_ANALYSES_DATA", _root / "data")) / "fig3")
print("working directory:", os.getcwd())


# GPU check

In [ ]:
import os
import torch

# Pick a GPU with CUDA_VISIBLE_DEVICES before starting Jupyter if the machine has several
print(torch.cuda.device_count())
if torch.cuda.device_count() > 0:
    print(torch.cuda.get_device_name(0))
else:
    print("no GPU visible: training needs one")

1
NVIDIA H200


In [ ]:
try:
    from cellpose import models, io, train, core
    from natsort import natsorted
    import tifffile as tiff
    print('Modules imported successfully')
except RuntimeError as e:
    print(f"Error importing cellpose: {e}")
    print("This might be due to a numpy version mismatch. Try restarting the runtime and running the cells again.")
except ImportError as e:
    print(f"Error importing a required library: {e}")
    print("Please ensure all necessary libraries are installed.")

Modules imported successfully


In [ ]:
import numpy as np
import tifffile
from skimage.segmentation import find_boundaries
from skimage.measure import label, regionprops
from skimage.morphology import opening, closing, disk
from typing import Tuple, List
import tqdm
import json


In [ ]:
# Training function (the same in every training notebook)

import os
import time
from datetime import timedelta
import matplotlib.pyplot as plt
import numpy as np
from cellpose import models, train


def train_cellpose_model(
        train_data,
        train_labels,
        channels,
        model_save_path,
        n_epochs,
        learning_rate,
        weight_decay,
        model_name,
        batch_size,
        normalize,
        min_train_masks,
        momentum,
        rescale=True,
        SGD=True,
        compute_flows=True,
        use_GPU=True,
        return_model=False,
        val_data=None,
        val_labels=None,
        pretrained_model_path=None
):
    """Train (or fine-tune) one Cellpose model on in-memory slices and save it under
    `model_save_path/models/model_name`.

    If that folder already exists the model is loaded instead of retrained. `pretrained_model_path`
    selects fine-tuning from an existing checkpoint; otherwise the network starts from scratch
    (`model_type=None`). Training uses `cellpose.train.train_seg` with the given SGD settings.
    The training loss curve is saved as `<model_name>_train_loss.npy`, and the validation curve
    as `<model_name>_val_loss.npy` when `val_data` / `val_labels` are passed (they are, for
    monitoring only; model selection happens at the 3D level).
    """
    import os
    import numpy as np
    from cellpose import models, train

    # Ensure output directory exists
    if not os.path.exists(model_save_path):
        print('Model directory does not exist, creating directory...')
        os.makedirs(model_save_path)

    model_save_dir = os.path.join(model_save_path, 'models', model_name)
    if os.path.exists(model_save_dir):
        print(f'Model already exists. Loading instead of training: {model_name}')
        model = models.CellposeModel(
            gpu=use_GPU,
            pretrained_model=model_save_dir
        )
        return model if return_model else None

    # === Load previous model (for fine-tuning) OR train from scratch
    if pretrained_model_path:
        print(f"Fine-tuning from previous model: {pretrained_model_path}")
        model = models.CellposeModel(gpu=use_GPU, pretrained_model=pretrained_model_path)
    else:
        print("Training model from scratch.")
        model = models.CellposeModel(gpu=use_GPU, model_type=None)

    print(f"Training model: {model_name}")

    # === Train the model
    new_model_path, train_loss, val_loss = train.train_seg(
        net=model.net,
        train_data=train_data,
        train_labels=train_labels,
        channels=channels,
        save_path=model_save_path,
        n_epochs=n_epochs,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        batch_size=batch_size,
        normalize=normalize,
        min_train_masks=min_train_masks,
        model_name=model_name,
        momentum=momentum,
        rescale=rescale,
        SGD=SGD,
        compute_flows=compute_flows,
        test_data=val_data if val_data is not None else None,
        test_labels=val_labels if val_labels is not None else None
    )

    # Save training and validation loss curves
    np.save(os.path.join(model_save_path, "models", model_name + '_train_loss.npy'), train_loss)
    if val_data or val_labels:
        np.save(os.path.join(model_save_path, "models", model_name + '_val_loss.npy'), val_loss)

    print(f"Trained Model saved to: {model_save_path}")
    return model if return_model else None


# Helper Functions -- Loader

In [ ]:
def load_exported_dataset(dataset_root):
    """
    Load an exported dataset snapshot from disk.
    """

    dataset_root = Path(dataset_root)

    def load_split(split):
        img_dir = dataset_root / split / "images"
        mask_dir = dataset_root / split / "masks"

        imgs, masks = [], []

        img_files = sorted(img_dir.glob("*_img.tif"))
        for img_path in img_files:
            mask_path = mask_dir / img_path.name.replace("_img.tif", "_mask.tif")

            img = tifffile.imread(img_path)
            mask = tifffile.imread(mask_path)

            imgs.append(img)
            masks.append(mask)

        return imgs, masks

    train_imgs, train_masks = load_split("train")
    val_imgs, val_masks = load_split("val")
    test_imgs, test_masks = load_split("test")

    with open(dataset_root / "metadata.json") as f:
        metadata = json.load(f)

    return (
        train_imgs, train_masks,
        val_imgs, val_masks,
        test_imgs, test_masks,
        metadata
    )


In [ ]:
from pathlib import Path
import tifffile
import json

# the split exported by 1_dense_splits/prepare_dense_2d_splits.ipynb (chunked splitting, chunk size 10)
(
    train_imgs, train_masks,
    val_imgs, val_masks,
    test_imgs, test_masks,
    metadata
) = load_exported_dataset(
    "decomposed_2D/decomposed2D_benchmark-strict_splits/lateral_dataset_v1"
)

In [ ]:
# Train from scratch
model = train_cellpose_model(
    train_data=train_imgs,
    train_labels=train_masks,
    channels=[2,1],  # membrane, nucleus
    model_save_path="domain-specialized_lateral_models/rescale_true",   # output folder
    n_epochs=500,
    learning_rate=1e-2,
    weight_decay=1e-5,
    model_name="domain-specialized_lateral_500epoch_scratch_strict_rescale_True",
    batch_size=16,
    normalize=True,
    min_train_masks=0,
    momentum=0.9,
    rescale=True,  # Table S6: rescale on
    SGD=True,
    compute_flows=True,
    use_GPU=True,
    return_model=True,
    val_data=val_imgs,
    val_labels=val_masks,
    pretrained_model_path=None
)


 Training model from scratch.
Training model: domain-specialized_lateral_500epoch_scratch_strict_rescale_True


  0%|                                                                                                         | 0/1675 [00:00<?, ?it/s]empty masks!
empty masks!
empty masks!
empty masks!
empty masks!
empty masks!
empty masks!
empty masks!
empty masks!
empty masks!
empty masks!
empty masks!
empty masks!
empty masks!
empty masks!
empty masks!
empty masks!
empty masks!
empty masks!
empty masks!
  5%|████▉                                                                                           | 86/1675 [00:02<01:09, 23.00it/s]empty masks!
empty masks!
empty masks!
empty masks!
  6%|█████▎                                                                                          | 93/1675 [00:02<00:49, 31.88it/s]empty masks!
empty masks!
empty masks!
empty masks!
empty masks!
empty masks!
empty masks!
empty masks!
empty masks!
empty masks!
empty masks!
empty masks!
empty masks!
empty masks!
empty masks!
empty masks!
empty masks!
empty masks!
empty masks!
 19%|█████████████████▊             

 Trained Model saved to: domain-specialized_lateral_models/rescale_true
